In [1]:
import json
import psycopg2
from kafka import KafkaConsumer
from datetime import datetime
from models import Ride, ride_deserializer

server = 'localhost:9092'
topic_name = 'rides'

In [2]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    consumer_timeout_ms=10000,  # Stops blocking after 5 seconds if topic is empty
    value_deserializer=ride_deserializer
)

/tmp/ipykernel_32240/2948794548.py:1: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


In [3]:
conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [4]:
print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
            (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
            VALUES (%s, %s, %s, %s, %s)""", 
            (ride.PULocationID, ride.DOLocationID, ride.trip_distance, ride.total_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to rides and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...


In [ ]:
print("Reading messages...")
message_count = 0

for record in consumer:
    print(record.value)
    message_count += 1

print(f"Done! Processed {message_count} messages.")
# # for message in consumer:
# #     print(f"Offset {message.offset}: {message.value}")
# #     # Remove break if you want to print all 12 messages (0 to 11)
# #     break

# # record = next(consumer)
# # record.value